# Noisy Minimum-Norm Path-Form Inversion — Demo Notebook

This notebook builds toward a **noisy minimum-norm inversion** workflow in which measurements are **path integrals over a 3D Sobolev model space**.

We use:
- A **unit-cube domain** $\Omega = [0,1]^3$
- A **low-order `Sobolev3D` model space** assembled from `BoxLaplacianSpectrumProvider` and `BesselSobolevBox`
- **10 random straight-ray paths** with sources drawn from every boundary face *except* the top face ($z = z_{\max}$) and receivers placed on the top face

**Phase 1 (this notebook):** Domain, model space, and acquisition geometry.  
**Phase 2** (future): Path-integral forward operator (`PathIntegralFunctional`, `SOLAOperator3D`) and noisy synthetic data.  
**Phase 3** (future): Noisy minimum-norm inversion via `LinearMinimumNormInversion` with a data covariance model.  
**Phase 4** (future): Model comparison visualizations and concluding discussion.

> **Note:** `PathIntegralFunctional` requires a `Sobolev3D` domain (not a plain `Lebesgue3D`) due to the unboundedness of 1-D curve evaluations in $L^2(\Omega)$. This constraint is enforced by the API and is exploited in Phase 2.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf import EuclideanSpace

from pygeoinf3d import (
    BoxDomain,
    Lebesgue3D,
    Path3D,
    BoxLaplacianSpectrumProvider,
    BesselSobolevBox,
    Sobolev3D,
)

# ── Reproducibility ────────────────────────────────────────────────────────
np.random.seed(42)
rng = np.random.default_rng(42)

---
## 1. Unit-Cube Domain and Low-Order Sobolev3D Model Space

We build:
1. A **unit-cube `BoxDomain`** — the physical domain $\Omega = [0,1]^3$.
2. A **`Lebesgue3D` base space** — the $L^2(\Omega)$ Hilbert space.
3. A **`BoxLaplacianSpectrumProvider`** with a coarse Neumann truncation — provides the spectral skeleton for the Sobolev weight.
4. A **low-order `Sobolev3D`** space assembled from `BesselSobolevBox` operators $W = (\tau - \Delta)^{s/2}$ with order $s = 1.75$.

A Sobolev space of order $s > d/2 = 1.5$ ensures that point evaluation, and therefore path-integral forms, is bounded in 3D. We choose a deliberately low but admissible order $s = 1.75$ so later inversion remains smooth without becoming overly expensive.

In [ ]:
# ── 1. Domain ─────────────────────────────────────────────────────────────
box        = BoxDomain(np.zeros(3), np.ones(3))
base_space = Lebesgue3D(box)

# ── 2. Spectral provider (coarse truncation — keeps Phase 1 fast) ──────────
TRUNCATION = (8, 8, 8)   # 8^3 = 512 Neumann modes
SHIFT      = 1.0         # ensures invertibility at the zero Laplacian mode
ORDER      = 1.75        # choose s > d/2 = 1.5 for bounded path evaluations

provider = BoxLaplacianSpectrumProvider(box, truncation=TRUNCATION, bc_type="neumann")

# ── 3. Sobolev3D model space ───────────────────────────────────────────────
W = BesselSobolevBox(base_space, provider, shift=SHIFT, order=ORDER)
W_inv = BesselSobolevBox(base_space, provider, shift=SHIFT, order=-ORDER)
sobolev = Sobolev3D(base_space, W, W_inv)

print(f"Domain:                {box!r}")
print(f"Spectral truncation:   {TRUNCATION}  →  {provider.dim} modes")
print(f"Sobolev order:         s = {ORDER}")
print(f"Weight shift:          τ = {SHIFT}")
print(f"Base space type:       {type(base_space).__name__}")
print(f"Model space type:      {type(sobolev).__name__}")

---
## 2. Acquisition Geometry

We model a surface-to-depth acquisition survey:

- **Sources** are placed on the **five non-top boundary faces** ($x_{\min}$, $x_{\max}$, $y_{\min}$, $y_{\max}$, $z_{\min}$) — representing side-wall and bottom illumination.
- **Receivers** are placed on the **top face** ($z = z_{\max} = 1$) — representing surface sensors.

Each source–receiver pair defines a **straight ray** (zero-offset `Path3D.from_polyline` with two waypoints).

### Face sampling

For a box with side lengths $L_x = L_y = L_z = 1$ all non-top faces have equal area $1 \times 1 = 1$, so sources are drawn with uniform probability across the five faces.  In general one would weight by face area.

In [ ]:
N_PATHS = 20
lo, hi = box.bounding_box().low, box.bounding_box().high
Lx, Ly, Lz = hi - lo

# ── Source sampling: all faces except z_max ────────────────────────────────
_face_labels = ["x_min", "x_max", "y_min", "y_max", "z_min"]
_face_areas = np.array([
    Ly * Lz,   # x_min
    Ly * Lz,   # x_max
    Lx * Lz,   # y_min
    Lx * Lz,   # y_max
    Lx * Ly,   # z_min
])
_face_probs = _face_areas / _face_areas.sum()

_face_indices = rng.choice(len(_face_labels), size=N_PATHS, p=_face_probs)
source_face_names = [_face_labels[i] for i in _face_indices]

sources = np.empty((N_PATHS, 3))
for k, fi in enumerate(_face_indices):
    face = _face_labels[fi]
    if face == "x_min":
        sources[k] = [lo[0], rng.uniform(lo[1], hi[1]), rng.uniform(lo[2], hi[2])]
    elif face == "x_max":
        sources[k] = [hi[0], rng.uniform(lo[1], hi[1]), rng.uniform(lo[2], hi[2])]
    elif face == "y_min":
        sources[k] = [rng.uniform(lo[0], hi[0]), lo[1], rng.uniform(lo[2], hi[2])]
    elif face == "y_max":
        sources[k] = [rng.uniform(lo[0], hi[0]), hi[1], rng.uniform(lo[2], hi[2])]
    else:  # z_min
        sources[k] = [rng.uniform(lo[0], hi[0]), rng.uniform(lo[1], hi[1]), lo[2]]

# ── Receiver sampling: top face (z = z_max) ───────────────────────────────
receivers = np.column_stack([
    rng.uniform(lo[0], hi[0], size=N_PATHS),
    rng.uniform(lo[1], hi[1], size=N_PATHS),
    np.full(N_PATHS, hi[2]),
])

# ── Straight-ray paths ────────────────────────────────────────────────────
paths = [
    Path3D.from_polyline(np.array([sources[i], receivers[i]]), domain=box)
    for i in range(N_PATHS)
]

---
## 3. Geometry Summary

The table below gives per-path diagnostics: source face, arc length, and endpoint coordinates.  We verify that:
- all source $z$-coordinates satisfy $z < z_{\max}$ (sources not on the top face), and
- all receiver $z$-coordinates equal $z_{\max}$.

In [ ]:
print(f"Box domain:            {box!r}")
print(f"Spectral modes:        {provider.dim}  (truncation {TRUNCATION})")
print(f"Sobolev order:         s = {ORDER},  shift τ = {SHIFT}")
print(f"Number of paths:       {N_PATHS}")
print()
print(f"{'Path':>5}  {'Source face':>10}  {'Arc length':>10}  "
      f"{'Source (x,y,z)':>24}  {'Receiver (x,y,z)':>26}")
print("-" * 85)
for i, (p, face) in enumerate(zip(paths, source_face_names)):
    sx, sy, sz = sources[i]
    rx, ry, rz = receivers[i]
    print(
        f"  {i+1:>3d}  {face:>10s}  {p.arclength:>10.4f}  "
        f"  ({sx:.3f}, {sy:.3f}, {sz:.3f})    ({rx:.3f}, {ry:.3f}, {rz:.3f})"
    )

# Sanity checks
assert all(sources[i, 2] < hi[2] for i in range(N_PATHS)), \
    "At least one source is on the top face — check sampling!"
assert all(abs(receivers[i, 2] - hi[2]) < 1e-12 for i in range(N_PATHS)), \
    "At least one receiver is not on z_max!"

print()
print("✓  All sources have z < z_max  (not on the top face).")
print("✓  All receivers lie on z = z_max.")

---
## 4. Interactive 3D Path Visualization

The cell below uses `plot_paths3d_interactive` (Plotly-backed) to render the 10 acquisition paths inside the unit cube.  Each label records which boundary face the source belongs to.

If Plotly is not installed, the cell prints an informative message instead of raising an error.  Install the optional dependency with:

```
pip install plotly
```

In [ ]:
try:
    from pygeoinf3d.plot import plot_paths3d_interactive

    labels = [
        f"path {i + 1}  (src: {source_face_names[i]})"
        for i in range(N_PATHS)
    ]
    fig = plot_paths3d_interactive(
        paths,
        labels=labels,
        n_samples=120,
        show_domain_box=True,
        show_markers=True,
        show_plot=True,
        title=(
            "Acquisition geometry — 10 straight rays\n"
            "(sources on all faces except z_max, receivers on z_max)"
        ),
    )

except ImportError:
    print(
        "Interactive path visualization requires Plotly, which is not installed.\n"
        "Install it with:  pip install plotly\n"
        "Then re-run this cell."
    )

## True model

The "ground truth" slowness field is a sum of three Gaussian blobs embedded in the unit cube:

$$u(\mathbf{x}) = \sum_{k=1}^{3} A_k \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{c}_k\|^2}{2\sigma_k^2}\right)$$

The function is stored as a `Function3D` with a callable evaluator and can be evaluated at any point in $[0,1]^3$.

In [ ]:
from pygeoinf3d import Function3D

centres    = [(0.25, 0.25, 0.25), (0.75, 0.50, 0.75), (0.50, 0.75, 0.25)]
amplitudes = [1.0, 0.8, 0.6]
scales     = [0.15, 0.12, 0.18]

def true_model_callable(x):
    x = np.atleast_2d(x)          # ensure shape (n, 3)
    result = np.zeros(x.shape[0])
    for centre, amp, scale in zip(centres, amplitudes, scales):
        diff = x - np.array(centre)
        result += amp * np.exp(-0.5 * np.sum(diff**2, axis=1) / scale**2)
    return result                  # shape (n,) for batched, or (1,) for single point

u_true = Function3D(box, evaluate_callable=true_model_callable, name="true model")

val = true_model_callable(np.array([[0.25, 0.25, 0.25]]))[0]
print(f"u_true name : {u_true.name}")
print(f"u_true(0.25, 0.25, 0.25) = {val:.6f}")

## Forward operator

Each path integral is encoded as a `PathIntegralFunctional`, which **requires a `Sobolev3D` domain** (path-integral representers need Sobolev regularity). The 10 forms are stacked into a `SOLAOperator3D` — a `LinearOperator` mapping $\mathcal{H}_W \to \mathbb{R}^{10}$.

In [ ]:
from pygeoinf3d.spaces.forms import PathIntegralFunctional
from pygeoinf3d.operators.sola import SOLAOperator3D

forms = [PathIntegralFunctional(sobolev, p, n_quad=300) for p in paths]
D = EuclideanSpace(N_PATHS)
G = SOLAOperator3D(sobolev, D, forms)

print(f"G domain   : {G.domain}")
print(f"G codomain : {G.codomain}")

## Synthetic data

Clean data are the exact path integrals of `u_true`. Gaussian noise with $\sigma = 2\%$ of the data range is added to simulate realistic measurement errors. The noise model and forward problem are constructed here for use in Phase 3.

In [ ]:
from pygeoinf import GaussianMeasure, LinearForwardProblem

# --- clean data ---
d_clean = G(u_true)           # shape (N_PATHS,)

# --- noise level ---
data_range = np.max(d_clean) - np.min(d_clean)
sigma = 0.02 * data_range if data_range > 1e-12 else 0.01

# --- noisy observations ---
noise   = rng.normal(0, sigma, size=N_PATHS)
d_noisy = d_clean + noise

# --- noise model ---
noise_measure = GaussianMeasure.from_covariance_matrix(
    G.codomain, sigma**2 * np.eye(N_PATHS)
)

# --- forward problem (for Phase 3) ---
forward_problem = LinearForwardProblem(G, data_error_measure=noise_measure)

snr = 10 * np.log10(np.mean(d_clean**2) / sigma**2) if sigma > 0 else float("inf")
print(f"d_clean : {np.round(d_clean, 5)}")
print(f"sigma   : {sigma:.6f}")
print(f"SNR     : {snr:.1f} dB")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_PATHS)
width = 0.35
ax.bar(x - width/2, d_clean, width, label="clean", alpha=0.8)
ax.bar(x + width/2, d_noisy, width, label="noisy", alpha=0.8)
ax.set_xlabel("Path index")
ax.set_ylabel("Path integral value")
ax.set_title("Clean vs noisy path-integral measurements")
ax.legend()
ax.set_xticks(x)
ax.set_xticklabels([f"Path {i}" for i in range(N_PATHS)], rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Noisy minimum-norm inversion

`LinearMinimumNormInversion` has two code paths:

- **With** `data_error_measure`: applies the discrepancy principle via model-space Tikhonov normal equations $(G^\top C_d^{-1} G + \alpha I)u = G^\top C_d^{-1} d$. This requires the model space to support a matrix representation (`from_components`), which `Sobolev3D` does not (it is basis-free).
- **Without** `data_error_measure`: uses the data-space formula $u = G^\top (G G^\top)^{-1} d$, where $G G^\top$ is the small $N \times N$ = 10×10 Gram matrix in `EuclideanSpace(10)`. This path is fully tractable.

We therefore pass a plain `LinearForwardProblem(G)` to force the data-space path. `EigenSolver(galerkin=True, rtol=1e-12)` pseudo-inverts the 10×10 Gram matrix, and $G^\top$ maps the solution back to the Sobolev model space.

Note: this gives the **undamped** minimum-norm solution — it exactly interpolates the noisy data (see diagnostics below). Regularisation comes from the Sobolev inner product itself: among all exactly-fitting functions, the minimum-norm one is as smooth as possible.

In [ ]:
from pygeoinf import LinearMinimumNormInversion, EigenSolver, LinearForwardProblem

# Use a plain forward problem (no noise model) to force the data-space path:
#   u = G^T (G G^T)^{-1} d
# where G G^T is a tractable 10×10 operator on EuclideanSpace(10).
plain_fp = LinearForwardProblem(G)
inversion = LinearMinimumNormInversion(plain_fp)
inverse_op = inversion.minimum_norm_operator(EigenSolver(galerkin=True, rtol=1e-12, parallel=True, n_jobs=12))
u_hat = inverse_op(d_noisy)

pt = np.array([0.25, 0.25, 0.25])
print(f"Recovered model type: {type(u_hat).__name__}")
print(f"u_hat(0.25,0.25,0.25)  = {np.asarray(u_hat(pt)).ravel()[0]:.6f}")
print(f"u_true(0.25,0.25,0.25) = {np.asarray(u_true(pt)).ravel()[0]:.6f}")

In [ ]:
# Data residual diagnostics
d_hat = G(u_hat)
residual = d_noisy - d_hat
chi_sq_fit = np.dot(residual, residual) / sigma**2
chi_sq_noise = np.dot(d_noisy - d_clean, d_noisy - d_clean) / sigma**2
rel_misfit = np.linalg.norm(residual) / np.linalg.norm(d_noisy)

print(f"Fit residual norm:           {np.linalg.norm(residual):.6e}")
print(f"Chi-squared (fit vs noisy):  {chi_sq_fit:.4f}  (expected ≈ 0 — exact interpolation)")
print(f"Chi-squared (noise only):    {chi_sq_noise:.4f}  (expected ≈ {N_PATHS} — noise realisation)")
print(f"Relative data misfit:        {rel_misfit:.6e}")
print(f"Degrees of freedom:          {N_PATHS}")

A chi-squared statistic near **zero** (fit vs noisy) confirms that the undamped minimum-norm solution exactly interpolates the noisy data. For a massively underdetermined system (10 measurements, 512 spectral modes), this is the **mathematically expected behaviour** of the minimum-norm pseudoinverse — not a pathology. The Sobolev inner product regularises in model space (enforcing spatial smoothness) even when the data constraint is satisfied exactly: among all functions that exactly fit the 10 noisy path integrals, the minimum Sobolev-norm one is selected.

The chi-squared of the noise realisation (noise only, ≈ $N_\text{paths}$) serves as a reference: it shows how much noise we absorbed into the fit. The discrepancy principle would seek a regularised solution with $\chi^2_\text{fit} \approx N_\text{paths}$, explicitly trading off data fit against model smoothness. Achieving this for a basis-free `Sobolev3D` model space requires a matrix-free discrepancy-principle implementation (e.g., iterative solver with explicit tolerance and augmented `Sobolev3D` in-place operations), which is a natural next step for this codebase.

## Model comparison

The interactive slider below shows three panels with a shared $z$-axis slider:

1. **True model** — the 3-Gaussian ground truth.
2. **Recovered (min-norm)** — the minimum-norm solution fitted to the 10 noisy path integrals.
3. **Total sensitivity** — the Riesz representative in $\mathcal{H}_W$ of the total path-integral form $\sum_{i=1}^{N} \ell_i$, computed as $G^*\mathbf{1} = G^*(1, \ldots, 1)^\top$. This shows where the 10 rays collectively illuminate the model space: regions with high sensitivity are well-constrained by the data, while regions with low sensitivity are invisible to the measurement set.

In [ ]:
plot_paths3d_interactive(paths[1], show_domain_box=True, show_markers=True, title="First path (straight ray)", show_plot=False)

In [ ]:
from pygeoinf3d.plot import plot_function3d_axis_slider
plot_function3d_axis_slider((G.adjoint)(D.basis_vector(1)), n_slices=10, resolution=50)

In [ ]:
try:
    from pygeoinf3d.plot import compare_function3d_axis_sliders

    # Total sensitivity: G^*(1) — Riesz representative in Sobolev3D of the
    # sum of all 10 path-integral forms.  Illuminated regions have large
    # |sensitivity|; the null space of G^T is invisible to the data.
    total_sensitivity = G.adjoint(np.ones(N_PATHS))

    comparison_fig = compare_function3d_axis_sliders(
        [u_true, u_hat, total_sensitivity],
        axis="z",
        n_slices=20,
        resolution=60,
        show_plot=False,
        titles=["True model", "Recovered (min-norm)", "Total sensitivity"],
        colorscale="RdBu_r",
        width=1200,
        height=400,
    )
    comparison_fig.show()
except ImportError as exc:
    print(
        "Interactive comparison unavailable. Install pygeoinf3d[interactive] "
        f"to enable Plotly-based slicing. ({exc})"
    )

## Summary and conclusions

This notebook demonstrated the complete minimum-norm inversion pipeline for a 3D geophysical model on a unit-cube geometry. The model space is `Sobolev3D` with smoothing order $s = 1.75$, which satisfies the Sobolev embedding criterion $s > d/2 = 1.5$ required for `PathIntegralFunctional` to be a bounded linear form on the model space; below this threshold, function evaluation along paths is not well-defined in an $L^2$ Hilbert space sense. The inner product is supplied by `BesselSobolevBox`, which weights each spectral mode by a shifted Bessel-type eigenvalue $(\lambda_k + \kappa^2)^s$, making smoother functions cheaper in norm and thereby encoding a spatial-smoothness prior.

With $N = 10$ path-integral measurements and $512$ spectral modes in the truncated basis, the system is massively underdetermined: the minimum-norm solution exactly interpolates the 10 noisy data values (chi-squared ≈ 0), as confirmed by the data-residual diagnostics. The Sobolev inner product acts as an implicit regulariser — among all functions that fit the data exactly, the one with minimum Sobolev norm is selected — but 10 measurements are far too few to constrain the 512-dimensional spectral expansion, so sharp spatial features of the true model (the three Gaussian blobs) are not recovered accurately. The `data-space branch` ($u = G^\top (GG^\top)^{-1} d$) was used because `Sobolev3D` is basis-free and does not yet support the `from_components` interface needed for the model-space Tikhonov branch. Implementing the discrepancy principle — seeking a regularised solution with $\chi^2_\text{fit} \approx N$ rather than $\chi^2_\text{fit} \approx 0$ — would require either adding `Sobolev3D.from_components` to enable the Tikhonov normal equations, or a matrix-free iterative solver that explicitly controls the data misfit.